# 📊 India Unemployment Rate — Exploratory Data Analysis
**Internship Project | CodeAlpha**

This notebook analyses India's unemployment data from **May 2019 to October 2020**, covering the pre-COVID baseline, the COVID-19 lockdown shock, and the early recovery phase.

### Datasets Used
-  — Monthly state-level data with **Rural / Urban** split (May 2019 – Jun 2020)
-  — State-level data with **lat/lon & geographic zone** (Jan 2020 – Oct 2020)

### Analysis Flow
1. Load & Clean Data
2. Merge & Feature Engineering
3. Univariate Analysis
4. State-wise Analysis
5. Rural vs Urban Comparison
6. COVID-19 Impact Analysis
7. Recovery Analysis
8. Key Insights


## 1️⃣ Load & Clean Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

# ── Style ──────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "#0f1117",
    "axes.facecolor":   "#1a1d27",
    "axes.edgecolor":   "#3a3d4d",
    "axes.labelcolor":  "#e0e0e0",
    "xtick.color":      "#b0b0b0",
    "ytick.color":      "#b0b0b0",
    "text.color":       "#e0e0e0",
    "grid.color":       "#2a2d3d",
    "grid.alpha":        0.5,
    "font.family":      "DejaVu Sans",
    "axes.titlesize":    14,
    "axes.labelsize":    11,
})

ACCENT  = "#00d4ff"
ACCENT2 = "#ff6b6b"
ACCENT3 = "#ffd166"
ACCENT4 = "#06d6a0"
PALETTE = [ACCENT, ACCENT2, ACCENT3, ACCENT4, "#a29bfe", "#fd79a8"]
print("✅ Libraries loaded")

In [ ]:
# Load both datasets
df1 = pd.read_csv("code_alpha_task.csv")
df2 = pd.read_csv("Unemployment_Rate_upto_11_2020.csv")

# Strip whitespace from column names
df1.columns = df1.columns.str.strip()
df2.columns = df2.columns.str.strip()

print("Dataset 1 shape:", df1.shape)
print("Dataset 2 shape:", df2.shape)
df1.head()

In [ ]:
# ── Clean Dataset 1 ───────────────────────────────────────────────
print(f"Null values in df1:
{df1.isnull().sum()}
")
df1.dropna(inplace=True)                              # drop 14 corrupt rows
df1["Date"] = pd.to_datetime(df1["Date"].str.strip(), dayfirst=True)
df1["Area"] = df1["Area"].str.strip()

# Rename for convenience
df1.rename(columns={
    "Estimated Unemployment Rate (%)": "Unemployment_Rate",
    "Estimated Employed":              "Employed",
    "Estimated Labour Participation Rate (%)": "LPR"
}, inplace=True)

# ── Clean Dataset 2 ───────────────────────────────────────────────
df2["Date"] = pd.to_datetime(df2["Date"].str.strip(), dayfirst=True)
df2.rename(columns={
    "Estimated Unemployment Rate (%)": "Unemployment_Rate",
    "Estimated Employed":              "Employed",
    "Estimated Labour Participation Rate (%)": "LPR",
    "Region.1":                        "Zone"
}, inplace=True)

print("df1 date range:", df1["Date"].min().date(), "→", df1["Date"].max().date())
print("df2 date range:", df2["Date"].min().date(), "→", df2["Date"].max().date())
print("
✅ Data cleaned!")

## 2️⃣ Merge & Feature Engineering

In [ ]:
# ── Geo lookup from df2 (lat, lon, Zone per state) ────────────────
geo = df2[["Region","Zone","longitude","latitude"]].drop_duplicates("Region")

# ── Master timeline: df1 (pre-Feb 2020) + df2 (Feb 2020 onwards) ──
# Aggregate df1 across Rural/Urban
df1_agg = (
    df1.groupby(["Region","Date"])
       .agg(Unemployment_Rate=("Unemployment_Rate","mean"),
            Employed=("Employed","sum"),
            LPR=("LPR","mean"))
       .reset_index()
)

cutoff   = pd.Timestamp("2020-02-01")
df1_pre  = df1_agg[df1_agg["Date"] < cutoff].copy()
df2_post = df2[["Region","Date","Unemployment_Rate","Employed","LPR"]].copy()

master = pd.concat([df1_pre, df2_post], ignore_index=True)
master = master.merge(geo, on="Region", how="left")
master.sort_values(["Region","Date"], inplace=True)

# ── Phase labels ──────────────────────────────────────────────────
def phase(d):
    if d < pd.Timestamp("2020-03-01"): return "Pre-COVID"
    if d < pd.Timestamp("2020-07-01"): return "COVID Lockdown"
    return "Post-Lockdown"

master["Phase"] = master["Date"].apply(phase)
master["Month"] = master["Date"].dt.strftime("%b-%y")

print(f"Master dataset: {master.shape}")
print(f"Date range    : {master.Date.min().date()} → {master.Date.max().date()}")
master.sample(5)

## 3️⃣ Univariate Analysis
> Distribution of the three key numeric variables across the full dataset.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Distribution of Key Variables", fontsize=16, y=1.02)

pairs = [
    ("Unemployment_Rate", "Unemployment Rate (%)",  ACCENT),
    ("LPR",               "Labour Participation Rate (%)", ACCENT3),
    ("Employed",          "People Employed",         ACCENT4),
]

for ax, (col, label, color) in zip(axes, pairs):
    data = master[col].dropna()
    ax.hist(data, bins=28, color=color, alpha=0.35,
            edgecolor="#ffffff15", density=True)
    kde_x = np.linspace(data.min(), data.max(), 300)
    ax.plot(kde_x, stats.gaussian_kde(data)(kde_x), color=color, lw=2.5)
    ax.axvline(data.mean(),   color=ACCENT2, lw=1.8, linestyle="--",
               label=f"Mean  {data.mean():.1f}")
    ax.axvline(data.median(), color=ACCENT3, lw=1.8, linestyle=":",
               label=f"Median {data.median():.1f}")
    ax.set_title(label)
    ax.set_xlabel(label)
    ax.legend(fontsize=9, framealpha=0.2)
    ax.grid(True, axis="y")

plt.tight_layout()
plt.show()

## 4️⃣ National Unemployment Trend  (Full 18-Month Timeline)
> The master dataset gives us a continuous view from **May 2019 → Oct 2020** — not possible with either dataset alone.

In [ ]:
national = master.groupby("Date")[["Unemployment_Rate","LPR"]].mean().reset_index()

fig, ax = plt.subplots(figsize=(14, 5))

ax.fill_between(national["Date"], national["Unemployment_Rate"],
                alpha=0.15, color=ACCENT)
ax.plot(national["Date"], national["Unemployment_Rate"],
        color=ACCENT, lw=2.5, marker="o", ms=4, label="Unemployment Rate (%)")
ax.plot(national["Date"], national["LPR"],
        color=ACCENT3, lw=2, linestyle="--", label="Labour Participation Rate (%)")

# Annotate lockdown band
ax.axvspan(pd.Timestamp("2020-03-25"), pd.Timestamp("2020-06-01"),
           alpha=0.12, color=ACCENT2, label="Lockdown Period")
ax.axvline(pd.Timestamp("2020-03-25"), color=ACCENT2, lw=1.2, linestyle=":")
ax.text(pd.Timestamp("2020-04-04"), national["Unemployment_Rate"].max()*0.93,
        "COVID-19
Lockdown", color=ACCENT2, fontsize=9, fontweight="bold")

ax.set_title("India — National Unemployment Rate  |  May 2019 → Oct 2020", pad=15)
ax.set_xlabel("Month")
ax.set_ylabel("Rate (%)")
ax.legend(framealpha=0.2)
ax.grid(True, axis="y")
plt.tight_layout()
plt.show()

pre  = national[national["Date"] < "2020-03-01"]["Unemployment_Rate"].mean()
peak = national["Unemployment_Rate"].max()
oct  = national[national["Date"] >= "2020-10-01"]["Unemployment_Rate"].mean()
print(f"Pre-COVID avg : {pre:.1f}%")
print(f"Lockdown peak : {peak:.1f}%  (+{peak-pre:.1f} pts)")
print(f"Oct-2020 avg  : {oct:.1f}%")

## 5️⃣ State-wise Analysis

In [ ]:
state_avg = master.groupby("Region")["Unemployment_Rate"].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Top 10 worst states ────────────────────────────────────────────
ax = axes[0]
top10 = state_avg.head(10)
colors = [ACCENT2 if v > 20 else ACCENT if v > 12 else ACCENT4 for v in top10.values]
bars = ax.barh(top10.index[::-1], top10.values[::-1],
               color=colors[::-1], edgecolor="#ffffff10", height=0.65)
for bar, val in zip(bars, top10.values[::-1]):
    ax.text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
            f"{val:.1f}%", va="center", fontsize=9)
ax.axvline(state_avg.mean(), color=ACCENT3, lw=1.5, linestyle="--")
ax.text(state_avg.mean()+0.2, 0.3,
        f"National
avg {state_avg.mean():.1f}%", color=ACCENT3, fontsize=8)
ax.set_title("Top 10 States — Highest Avg Unemployment")
ax.set_xlabel("Average Unemployment Rate (%)")
ax.grid(True, axis="x")

# ── Bottom 10 best states ─────────────────────────────────────────
ax2 = axes[1]
bot10 = state_avg.tail(10)
bars2 = ax2.barh(bot10.index, bot10.values,
                 color=ACCENT4, edgecolor="#ffffff10", height=0.65)
for bar, val in zip(bars2, bot10.values):
    ax2.text(bar.get_width()+0.1, bar.get_y()+bar.get_height()/2,
             f"{val:.1f}%", va="center", fontsize=9)
ax2.set_title("Top 10 States — Lowest Avg Unemployment")
ax2.set_xlabel("Average Unemployment Rate (%)")
ax2.grid(True, axis="x")

fig.suptitle("State-wise Unemployment Comparison", fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
# State × Month heatmap
pivot = master.pivot_table(index="Region", columns="Date",
                            values="Unemployment_Rate", aggfunc="mean")
pivot.columns = [d.strftime("%b-%y") for d in pivot.columns]

fig, ax = plt.subplots(figsize=(18, 9))
sns.heatmap(pivot, cmap="YlOrRd", linewidths=0.3, linecolor="#1a1d27",
            ax=ax, cbar_kws={"label": "Unemployment Rate (%)", "shrink": 0.5},
            annot=False)
ax.set_title("State × Month Unemployment Heatmap  |  May 2019 → Oct 2020",
             pad=15, fontsize=15)
ax.set_xlabel("Month")
ax.set_ylabel("State / UT")
ax.tick_params(axis="x", rotation=45, labelsize=8)
plt.tight_layout()
plt.show()

## 6️⃣ Rural vs Urban Comparison
> Dataset 1 is the only source with this breakdown — a unique dimension in our analysis.

In [ ]:
ru = df1.groupby(["Date","Area"])["Unemployment_Rate"].mean().reset_index()
rural = ru[ru["Area"]=="Rural"]
urban = ru[ru["Area"]=="Urban"]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Line trend
ax = axes[0]
ax.fill_between(rural["Date"], rural["Unemployment_Rate"], alpha=0.12, color=ACCENT4)
ax.fill_between(urban["Date"], urban["Unemployment_Rate"], alpha=0.12, color=ACCENT2)
ax.plot(rural["Date"], rural["Unemployment_Rate"],
        color=ACCENT4, lw=2.2, marker="o", ms=3, label="Rural")
ax.plot(urban["Date"], urban["Unemployment_Rate"],
        color=ACCENT2, lw=2.2, marker="o", ms=3, label="Urban")
ax.axvspan(pd.Timestamp("2020-03-25"), pd.Timestamp("2020-06-01"),
           alpha=0.10, color=ACCENT2)
ax.set_title("Monthly Trend — Rural vs Urban")
ax.set_ylabel("Unemployment Rate (%)")
ax.legend(framealpha=0.2)
ax.grid(True, axis="y")

# Box plot
ax2 = axes[1]
data_r = df1[df1["Area"]=="Rural"]["Unemployment_Rate"]
data_u = df1[df1["Area"]=="Urban"]["Unemployment_Rate"]
bp = ax2.boxplot([data_r, data_u], patch_artist=True,
                  medianprops=dict(color="#fff", lw=2),
                  whiskerprops=dict(color="#b0b0b0"),
                  capprops=dict(color="#b0b0b0"),
                  flierprops=dict(marker="o", ms=3, color="#b0b0b0"))
bp["boxes"][0].set_facecolor(ACCENT4+"66")
bp["boxes"][1].set_facecolor(ACCENT2+"66")
ax2.set_xticklabels(["Rural","Urban"])
ax2.set_title("Distribution — Rural vs Urban")
ax2.set_ylabel("Unemployment Rate (%)")
ax2.grid(True, axis="y")

fig.suptitle("Rural vs Urban Unemployment", fontsize=15)
plt.tight_layout()
plt.show()

print(f"Rural avg : {data_r.mean():.2f}%")
print(f"Urban avg : {data_u.mean():.2f}%")
print(f"Urban was {data_u.mean()-data_r.mean():.2f}% higher on average")

## 7️⃣ COVID-19 Impact Analysis

In [ ]:
phase_state = master.groupby(["Region","Phase"])["Unemployment_Rate"].mean().reset_index()
pivot_phase = phase_state.pivot(index="Region", columns="Phase",
                                 values="Unemployment_Rate").fillna(0)

cols = [c for c in ["Pre-COVID","COVID Lockdown","Post-Lockdown"]
        if c in pivot_phase.columns]
pivot_phase = pivot_phase[cols]
pivot_phase["Spike"] = pivot_phase.get("COVID Lockdown",0) - pivot_phase.get("Pre-COVID",0)
pivot_phase.sort_values("Spike", ascending=False, inplace=True)

fig, ax = plt.subplots(figsize=(13, 7))
x = np.arange(len(pivot_phase))
w = 0.27
phase_colors = [ACCENT4, ACCENT2, ACCENT]

for i, col in enumerate(cols):
    ax.bar(x + i*w, pivot_phase[col], width=w,
           color=phase_colors[i], label=col, alpha=0.88,
           edgecolor="#ffffff10")

ax.set_xticks(x + w)
ax.set_xticklabels(pivot_phase.index, rotation=45, ha="right", fontsize=8)
ax.set_title("COVID-19 Impact by State — Pre / During / Post Lockdown", pad=15)
ax.set_ylabel("Average Unemployment Rate (%)")
ax.legend(framealpha=0.2)
ax.grid(True, axis="y")
plt.tight_layout()
plt.show()

## 8️⃣ Recovery Analysis  (Jul – Oct 2020)
> Only possible using Dataset 2, which extends to October 2020.

In [ ]:
recovery = pivot_phase.copy()
if "Post-Lockdown" in recovery.columns and "Pre-COVID" in recovery.columns:
    recovery["Recovery_%"] = (
        (recovery["COVID Lockdown"] - recovery["Post-Lockdown"]) /
        (recovery["COVID Lockdown"] - recovery["Pre-COVID"] + 0.001) * 100
    ).clip(0, 100)
    recovery.sort_values("Spike", ascending=False, inplace=True)
    top_hit = recovery.head(12)

    fig, ax = plt.subplots(figsize=(12, 6))
    rec_colors = [ACCENT4 if v>=70 else ACCENT3 if v>=40 else ACCENT2
                  for v in top_hit["Recovery_%"]]
    bars = ax.barh(top_hit.index[::-1], top_hit["Recovery_%"][::-1],
                   color=rec_colors[::-1], edgecolor="#ffffff10", height=0.6)
    for bar, val in zip(bars, top_hit["Recovery_%"][::-1]):
        ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
                f"{val:.0f}%", va="center", fontsize=9)

    ax.axvline(70, color=ACCENT4, lw=1.5, linestyle="--")
    ax.text(71, 0.3, "70% recovery
threshold", color=ACCENT4, fontsize=8)
    ax.set_title("Recovery Rate by Oct 2020  (Most COVID-impacted States)", pad=15)
    ax.set_xlabel("Recovery Rate (%) from Lockdown Peak")
    ax.set_xlim(0, 115)

    patches = [
        mpatches.Patch(color=ACCENT4, label="Strong (≥70%)"),
        mpatches.Patch(color=ACCENT3, label="Moderate (40–70%)"),
        mpatches.Patch(color=ACCENT2, label="Weak (<40%)"),
    ]
    ax.legend(handles=patches, framealpha=0.2)
    ax.grid(True, axis="x")
    plt.tight_layout()
    plt.show()

## 9️⃣ Zone-wise Trend
> Geographic zones from Dataset 2 reveal regional patterns across India.

In [ ]:
zone_time = (master.dropna(subset=["Zone"])
             .groupby(["Zone","Date"])["Unemployment_Rate"].mean()
             .reset_index())

zones    = zone_time["Zone"].unique()
zcolors  = dict(zip(zones, PALETTE))

fig, ax = plt.subplots(figsize=(13, 5))
for zone in zones:
    d = zone_time[zone_time["Zone"]==zone]
    ax.plot(d["Date"], d["Unemployment_Rate"],
            color=zcolors[zone], lw=2.2, marker="o", ms=4, label=zone)

ax.axvspan(pd.Timestamp("2020-03-25"), pd.Timestamp("2020-06-01"),
           alpha=0.10, color=ACCENT2)
ax.set_title("Zone-wise Monthly Unemployment  |  Feb 2020 → Oct 2020", pad=15)
ax.set_ylabel("Unemployment Rate (%)")
ax.legend(framealpha=0.2, title="Zone")
ax.grid(True, axis="y")
plt.tight_layout()
plt.show()

## 🔟 Correlation & Scatter Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Correlation heatmap
corr = master[["Unemployment_Rate","Employed","LPR"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            linewidths=1, linecolor="#1a1d27", ax=axes[0],
            annot_kws={"size":13, "weight":"bold"})
axes[0].set_title("Correlation Matrix")

# LPR vs Unemployment scatter
ax2 = axes[1]
scatter = master.dropna(subset=["Zone"])
for zone in scatter["Zone"].unique():
    d = scatter[scatter["Zone"]==zone]
    ax2.scatter(d["LPR"], d["Unemployment_Rate"],
                color=zcolors.get(zone, ACCENT), alpha=0.5, s=25, label=zone)

x = scatter["LPR"]; y = scatter["Unemployment_Rate"]
m, b, r, *_ = stats.linregress(x, y)
xl = np.linspace(x.min(), x.max(), 100)
ax2.plot(xl, m*xl+b, color=ACCENT3, lw=2, linestyle="--",
         label=f"Trend  r={r:.2f}")
ax2.set_title("LPR vs Unemployment Rate")
ax2.set_xlabel("Labour Participation Rate (%)")
ax2.set_ylabel("Unemployment Rate (%)")
ax2.legend(framealpha=0.2, fontsize=8, title="Zone")
ax2.grid(True)

plt.tight_layout()
plt.show()

## 📝 Key Insights

| # | Insight |
|---|---|
| 1 | India's national unemployment jumped from ~9.5% (pre-COVID) to **23.2% peak** during April–May 2020 lockdown |
| 2 | By October 2020, unemployment had largely recovered to **~8%** — near the pre-COVID baseline |
| 3 | **Urban unemployment** (13.2%) was consistently higher than **Rural** (10.3%) — lockdowns disrupted formal employment more |
| 4 | **Tripura, Haryana & Jharkhand** had the highest average unemployment across the full period |
| 5 | **Meghalaya, Karnataka & Telangana** consistently maintained the lowest unemployment |
| 6 | Labour Participation Rate and Unemployment show a **negative correlation** — as LPR drops, unemployment rises |
| 7 | The **North and East zones** were hardest hit during lockdown; **South zone** recovered fastest |

### 🏆 What Makes This Project Stand Out
- Uses **two datasets merged** into a single 18-month master timeline
- Covers **three analytical dimensions**: time, geography (state + zone), and area type (rural/urban)
- Clearly tells the **COVID-19 story** from shock → peak → recovery
